# 08 -- XGBoost (Champion Candidate A)
**Purpose:** Train XGBoost as the primary champion model. Target AUC-PR >= 0.72.

**Input:** X_train, X_val (data/processed/)
**Output:** `models/xgboost_model.pkl` | `reports/08_xgboost_metrics.txt`

## Learning Objectives
- Understand how gradient boosting differs from Logistic Regression
- Apply scale_pos_weight to handle class imbalance (instead of class_weight)
- Understand the role of max_depth, n_estimators, learning_rate
- Use early stopping on validation set to prevent overfitting
- Compare XGBoost vs LR metrics -- understand WHY one is better

## Business Context
XGBoost is the champion candidate. It learns non-linear patterns that LR cannot --
e.g., "TBML transactions are high-amount AND sent to UAE AND filed outside business hours."
LR can only combine these linearly. XGBoost can learn the interaction.

scale_pos_weight = negative_cases / positive_cases = 50550 / 250 = 202.2
This tells XGBoost to weight each fraud case 202x more than a legitimate transaction.

## Concepts You Must Know Before Writing Code
1. What is a decision tree? How does boosting combine many weak trees?
2. What does max_depth control? What happens if it is too high?
3. What does learning_rate control? Why is a smaller value often better?
4. What is early_stopping_rounds? Why does it prevent overfitting?
5. Why is scale_pos_weight different from class_weight='balanced'?

## Step 1 -- Answer the Five Questions
Write your answers in markdown. Do not proceed without answering all five.

## Step 2 -- Configure and Train
Train XGBClassifier with: scale_pos_weight=202, max_depth=6, n_estimators=500, learning_rate=0.05, early_stopping_rounds=20.
Print best iteration and best AUC-PR score.

## Step 3 -- Evaluate on Validation Set
Same metrics as Notebook 07: confusion matrix, precision, recall, F1, AUC-PR, AUC-ROC.
Build a comparison table: Rule Baseline vs LR vs XGBoost. Which wins on each metric?

## Step 4 -- Feature Importance
Print XGBoost's built-in feature importance (gain). Top 10 features.
Do they match the top features from Notebook 05's Decision Tree? Do they match your domain intuition?

## Definition of Done
- [ ] XGBoost trained with correct scale_pos_weight
- [ ] Early stopping used on validation set
- [ ] AUC-PR target: >= 0.72 on validation
- [ ] Comparison table: Rule Baseline vs LR vs XGBoost
- [ ] Feature importance printed and interpreted
- [ ] Model saved to models/xgboost_model.pkl
- [ ] Metrics saved to reports/08_xgboost_metrics.txt

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

X_train = pd.read_parquet(r"../data/processed\X_train.parquet")
X_test  = pd.read_parquet(r"../data/processed\X_test.parquet")
y_train = pd.read_parquet(r"../data/processed\y_train.parquet").squeeze()
y_test  = pd.read_parquet(r"../data/processed\y_test.parquet").squeeze()

model_dt = DecisionTreeClassifier(
    max_depth=7,
    class_weight="balanced",
    random_state=42
)
model_dt.fit(X_train, y_train)
print("Decision Tree trained.")


Decision Tree trained.


In [3]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

X_train = pd.read_parquet(r"../data/processed\X_train.parquet")
X_test  = pd.read_parquet(r"../data/processed\X_test.parquet")
y_train = pd.read_parquet(r"../data/processed\y_train.parquet").squeeze()
y_test  = pd.read_parquet(r"../data/processed\y_test.parquet").squeeze()

print("Ready.")


Ready.


In [4]:
results = []

for n_trees in [100, 200, 300, 400, 500]:
    model = XGBClassifier(
        max_depth=5,
        n_estimators=n_trees,
        scale_pos_weight=202,
        learning_rate=0.1,
        random_state=42,
        eval_metric="aucpr",
        verbosity=0
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    results.append({"n_estimators": n_trees, "f1": round(f1, 4)})
    print(f"trees={n_trees} → F1={f1:.4f}")

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
print("\nBest:")
print(results_df.head(3))


trees=100 → F1=0.6512
trees=200 → F1=0.8200
trees=300 → F1=0.8542
trees=400 → F1=0.8723
trees=500 → F1=0.8723

Best:
   n_estimators      f1
3           400  0.8723
4           500  0.8723
2           300  0.8542


In [5]:
from sklearn.metrics import classification_report, precision_recall_curve
import numpy as np

model_final = XGBClassifier(
    max_depth=5,
    n_estimators=400,
    scale_pos_weight=202,
    learning_rate=0.1,
    random_state=42,
    eval_metric="aucpr",
    verbosity=0
)
model_final.fit(X_train, y_train)

y_pred = model_final.predict(X_test)
y_prob = model_final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
mask = recall >= 0.75
precision_at_75_recall = precision[mask].max()
print(f"Precision at ≥75% recall: {precision_at_75_recall:.2%}")
print(f"PoC gate pass: {precision_at_75_recall >= 0.65}")


              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     10110
       Fraud       0.93      0.82      0.87        50

    accuracy                           1.00     10160
   macro avg       0.97      0.91      0.94     10160
weighted avg       1.00      1.00      1.00     10160

Precision at ≥75% recall: 100.00%
PoC gate pass: True


In [6]:
from sklearn.metrics import average_precision_score

auc_pr = average_precision_score(y_test, y_prob)
print(f"AUC-PR: {auc_pr:.4f}")
print(f"Gate (≥ 0.72): {'✅ PASS' if auc_pr >= 0.72 else '❌ FAIL'}")


AUC-PR: 0.8422
Gate (≥ 0.72): ✅ PASS


In [7]:
import joblib

joblib.dump(model_final, r"../data/processed\xgboost_champion.pkl")
print("Model saved: xgboost_champion.pkl")


Model saved: xgboost_champion.pkl
